# M18 lab solution: Post-Retrieval Optimisation & Reranking

**Brief.** Precision stack: add a cross-encoder reranker over a top-50 hybrid retrieval and measure the gain at top-5, plus the latency budget it consumes.

Run with the `hf` group: `uv run --group hf jupyter lab` (the cross-encoder runs locally; the first run downloads it).

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

## 1. A corpus deep enough for top 50

At 500 characters the handbook has about 30 chunks, too few for a top-50 first stage. This lab splits at 200 characters (overlap 20), so the hybrid retriever has more than 50 candidates to rank.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
DEPTH, TOP = 50, 5
print(len(chunks), "chunks")

bm25 = BM25Retriever.from_documents(chunks, k=DEPTH)
dense = FAISS.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": DEPTH})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

## 2. Stage 1: top 50 hybrid

Recall at 50 is the ceiling for everything after it: the reranker only reorders what stage 1 returned.

In [ ]:
candidates = {g["id"]: texts(hybrid.invoke(g["question"]))[:DEPTH] for g in golden}
print("stage 1 at depth 50:", score(candidates))
print("stage 1 at top 5:  ", score({k: v[:TOP] for k, v in candidates.items()}))

## 3. Stage 2: cross-encoder reranking

`cross-encoder/ms-marco-MiniLM-L-6-v2` reads each question and candidate together and scores the pair. The top 5 after reranking is compared with the top 5 before it; reranking time is measured per query.

In [ ]:
import statistics

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker.predict([("warm up", "the first call loads the model")])

reranked, seconds = {}, []
for g in golden:
    cands = candidates[g["id"]]
    t0 = time.perf_counter()
    scores = reranker.predict([(g["question"], c) for c in cands])
    seconds.append(time.perf_counter() - t0)
    reranked[g["id"]] = [c for _, c in sorted(zip(scores, cands), key=lambda x: -x[0])][:TOP]

before, after = score({k: v[:TOP] for k, v in candidates.items()}), score(reranked)
latency = {"mean_ms": round(statistics.mean(seconds) * 1000, 1), "p50_ms": round(statistics.median(seconds) * 1000, 1),
           "p95_ms": round(statistics.quantiles(seconds, n=20)[18] * 1000, 1)}
print("before:", before, "\nafter: ", after, "\nrerank latency:", latency)

## Results

In [ ]:
import pandas as pd

print("run:", time.strftime("%Y-%m-%d"), "| embeddings:", EMBEDDING_NAME, "| reranker: ms-marco-MiniLM-L-6-v2 on this machine")
print(pd.DataFrame([{"stage": "hybrid top 5", **before}, {"stage": "hybrid top 50 + cross-encoder, top 5", **after}]).to_string(index=False))
print(latency)

## What this shows

- The gain at top 5 is bounded by recall at 50: if stage 1 never returned a span, no reranker can put it in the top 5.
- Rerank latency grows with the number of candidates, one forward pass per question and chunk pair; the p95 figure is the one to hold against a latency budget.
- MRR moves more than hit rate when the reranker mostly reorders evidence that was already in the top 5.